# Kyrgyz (kir) — Full Neural Pipeline with Cyrillic/Latin Support

Kyrgyz supports the full Stanza neural pipeline via the KTMU UD treebank (POS, lemma, depparse), Apertium FST morphology (Stable quality), and bidirectional Cyrillic↔Latin transliteration. NLLB-200 provides embeddings and translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('kir')

## 2. Script Detection and Cyrillic ↔ Latin Transliteration

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

cyrl = "Бишкек Кыргызстандын башкаласы."
print("Detected:", detect_script(cyrl))

t = Transliterator("kir", source=Script.CYRILLIC, target=Script.LATIN)
latin = t.transliterate(cyrl)
print("Latin:", latin)

## 3. Morphological Analysis (Apertium FST)

In [ ]:
nlp_morph = Pipeline(
    "kir",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
    script="Cyrl",
)
doc = nlp_morph("Мен мектепке барам.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

## 4. POS Tagging, Lemmatisation, and Dependency Parsing (KTMU treebank)

In [ ]:
nlp_parse = Pipeline(
    "kir",
    processors=["tokenize", "pos", "lemma", "depparse"],
    script="Cyrl",
)

doc = nlp_parse("Ала-Тоо тоолор Кыргызстанда жайгашкан.")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<20} {'Deprel'}")
print("-" * 60)
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<20} {w.deprel}")

## 5. Full Pipeline with CoNLL-U Export

In [ ]:
nlp_full = Pipeline(
    "kir",
    processors=["tokenize", "morph", "pos", "lemma", "depparse"],
    morph_backend="apertium",
    script="Cyrl",
)
doc = nlp_full("Кыргыз тили Кыргызстандын расмий тили болуп эсептелет.")
print(doc.to_conllu())

## 6. Embeddings and Translation

In [ ]:
turkicnlp.download("kir", processors=["translate"])
trans = Pipeline("kir", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("Кыргызстан — Борбордук Азиядагы мамлекет.")
print("EN:", doc.translation)